# LTAT.02.006 Andmeteaduse meetodid

## 9. praktikum "Markovi mudelid".

Tänases praktikumis läheb vaja Python'i paketti `hmmlearn`. Selle installimiseks valige üks nendest käskudest:

-   `conda install hmmlearn`
-   `conda install -c conda-forge hmmlearn`
-   `pip install hmmlearn`

[hmmlearn-i dokumentatsioon](https://hmmlearn.readthedocs.io/en/latest/api.html#hmmlearn-hmm)


In [1]:
!pip install hmmlearn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\Saskia\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from hmmlearn.hmm import CategoricalHMM

### 1. Peidetud markovi mudeli andmetest õppimine

Ülesanne põhineb YouTube'i videol:

[A friendly introduction to Bayes Theorem and Hidden Markov Models](https://www.youtube.com/watch?v=kqSzLo9fenk)

Antud on andmestik, milles on järjestikustel päevadel vaadeldud ilma (päikseline 🌞 vs vihmane 🌧️) ja inimese tuju (rõõmus 😊 vs kurb 😔). Andmepunktid moodustavad aegrea, st. nende järjekord on oluline. Andmestik on antud Pandas'e andmeraami kujul, mille indeksi veeru nimi on "Päev". Indeksi veerg ei ole tunnus, see on andmepunkti järjekorranumber.


In [3]:
päikseline = '🌞'
vihmane = '🌧️'
rõõmus = '😊'
kurb = '😔'

andmed = pd.DataFrame({
    "Ilm": [
        päikseline,
        päikseline,
        päikseline,
        päikseline,
        vihmane,
        vihmane,
        vihmane,
        päikseline,
        päikseline,
        päikseline,
        päikseline,
        vihmane,
        vihmane,
        päikseline,
        päikseline,
    ],
    "Tuju": [
        kurb,
        rõõmus,
        rõõmus,
        rõõmus,
        kurb,
        kurb,
        rõõmus,
        kurb,
        rõõmus,
        rõõmus,
        rõõmus,
        kurb,
        rõõmus,
        rõõmus,
        rõõmus
    ]
}, index=range(1, 16))

andmed.index.name = "Päev"

andmed

,Ilm,Tuju
Päev,,
1,🌞,😔
2,🌞,😊
3,🌞,😊
4,🌞,😊
5,🌧️,😔
6,🌧️,😔
7,🌧️,😊
8,🌞,😔
9,🌞,😊


Arvuta järgmised tõenäosused:

| Kirjeldus                                                               | Tähis                        |
| ----------------------------------------------------------------------- | ---------------------------- |
| Tõenäosus, et täna paistab päike                                        | P( 🌞 )                      |
| Tõenäosus, et täna sajab vihma                                          | P( 🌧️ )                      |
| Tõenäosus, et täna paistab päike, kui eilne ilm oli samuti päikseline   | P( 🌞 \| 🌞 )                |
| Tõenäosus, et täna sajab vihma, kui eilne ilm oli päikseline            | P( 🌧️ \| 🌞 )                |
| Tõenäosus, et täna sajab vihma/paistab päike, kui eilne ilm oli vihmane | P( 🌧️ \| 🌧️ ), P( 🌞 \| 🌧️ ) |
| Tõenäosus, et päikseline päev teeb inimese rõõmsaks/kurvaks             | P( 😊 \| 🌞 ), P( 😔 \| 🌞 ) |
| Tõenäosus, et vihmane päev teeb inimese rõõmsaks/kurvaks                | P( 😊 \| 🌧️ ), P( 😔 \| 🌧️ ) |

Kasulik on teada tingliku tõenäosuse valemit:

$$ P(A|B) = \frac{P(A,B)}{P(B)} $$

Tõenäosusi saab andmete põhjal arvutada tabeli ridu kokku lugedes.


In [6]:
p_päikseline = 10/15
p_vihmane = 5/15
p_päikseline_päikseline = 7/9
p_vihmane_päikseline = 2/9
p_päikseline_vihmane = 2/5
p_vihmane_vihmane = 3/5

p_rõõmus_päikseline = 8/10
p_kurb_päikseline = 2/10
p_rõõmus_vihmane = 2/5
p_kurb_vihmane = 3/5

Loo eelnevalt arvutatud tõenäosuste põhjal peidetud markovi mudel, kus peidetud olekuks on ilm ning vaatluseks on tuju. Genereeri sellest mudelist andmeid.


In [7]:
peidetud_olekud = np.array([päikseline, vihmane])
vaatlused = np.array([rõõmus, kurb])

hmm = CategoricalHMM(n_components = 2, init_params="", random_state=0)

# Algoleku tõenäosused
hmm.startprob_ = np.array([p_päikseline, p_vihmane])

# Üleminekutõenäosused
hmm.transmat_ = np.array([
	[p_päikseline_päikseline, p_vihmane_päikseline],
	[p_päikseline_vihmane, p_vihmane_vihmane]
])

# Väljastamise tõenäosused
hmm.emissionprob_ = np.array([
	[p_rõõmus_päikseline, p_kurb_päikseline],
	[p_rõõmus_vihmane, p_kurb_vihmane]
])


In [8]:
genereeritud_ahel = hmm.sample(10)
peidetud_olekute_indeksid = genereeritud_ahel[0].flatten()
vaatluste_indeksid = genereeritud_ahel[1]

print(peidetud_olekud[peidetud_olekute_indeksid])
print(vaatlused[vaatluste_indeksid])

['🌞' '🌞' '🌞' '🌧️' '🌞' '🌧️' '🌧️' '🌞' '🌧️' '🌧️']
['😊' '😊' '😊' '😊' '😔' '😔' '😔' '😊' '😊' '😔']


Vaatleme ühte juhuslikult valitud inimest viiel järjestikusel päeval. Iga päev helistame talle ning küsime, milline on tema tuju kuid me ei küsi ilma. Saame teada, et nendel päevadel oli tema tuju 😊, 😊, 😔, 😔, 😔, 😊. Milline oli neil päevil kõige tõenäolisem ilm eelmises punktis loodud Markovi mudeli järgi? Kasuta selleks Viterbi algoritmi.


In [11]:
rõõmus_i = 0
kurb_i = 1

vaadeldud_tuju = np.array([[rõõmus_i], [rõõmus_i], [kurb_i], [kurb_i], [kurb_i], [rõõmus_i]])
logprob, tõenäolisim_järjend = hmm.decode(vaadeldud_tuju, algorithm="viterbi")

print(peidetud_olekud[tõenäolisim_järjend])

print(logprob)

['🌞' '🌞' '🌧️' '🌧️' '🌧️' '🌞']
-6.300706437812082
